# Feasibility of a cleaned WLASL with variation-level classes

Two changes to WLASL are being considered:

1. **Fix the duplicate clips**: drop copies, and merge glosses that are the same sign (see
   [duplicate_clips.ipynb](./duplicate_clips.ipynb) and "Duplicate clips" in
   [WLASL_info.md](../../info/WLASL_info.md)).
2. **Classify variations, not glosses**: a class is a `(gloss, variation_id)` pair, since the
   variations of a gloss can be quite different signs (`before` variation 0 and 1).

This notebook counts what each would leave, on WLASL-2000's instances: how many classes, how many
instances per class and set, and where the labels are weakest. It builds no dataset.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.results import STASH_DIR_NAME, load_json
from src.results.dataset_analysis.duplicates import merge_signs, shared_clip_counts
from src.run_types import SPLIT_DIR

md5_stash_p = Path(STASH_DIR_NAME) / "clip_md5.json"
glosses = json.loads((SPLIT_DIR / "asl2000.json").read_text())
instances = pd.DataFrame(
    [
        {
            "video_id": i["video_id"],
            "gloss": g["gloss"],
            "set": i["split"],
            "source": i["source"],
            "signer_id": i["signer_id"],
            "variation_id": i["variation_id"],
        }
        for g in glosses
        for i in g["instances"]
    ]
)
instances["md5"] = instances.video_id.map(load_json(md5_stash_p))
len(instances)

21095

## Variations as classes

A class is a `(gloss, variation_id)` pair. `train`, `val` and `test` are its instances per set.

In [2]:
per_gloss = instances.groupby("gloss").variation_id.nunique()
print(f"{int((per_gloss > 1).sum())} of {len(per_gloss)} glosses have more than one variation:")
print(per_gloss.value_counts().sort_index().rename("glosses").to_frame().T.to_string())

def class_sizes(frame: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    return frame.groupby([*by, "set"]).size().unstack(fill_value=0).reindex(columns=["train", "val", "test"], fill_value=0)

variation_classes = class_sizes(instances, ["gloss", "variation_id"])
gloss_classes = class_sizes(instances, ["gloss"])
pd.DataFrame(
    {
        "classes": {"gloss": len(gloss_classes), "(gloss, variation)": len(variation_classes)},
        "median train": {"gloss": gloss_classes.train.median(), "(gloss, variation)": variation_classes.train.median()},
        "min train": {"gloss": gloss_classes.train.min(), "(gloss, variation)": variation_classes.train.min()},
        "train < 5": {"gloss": int((gloss_classes.train < 5).sum()), "(gloss, variation)": int((variation_classes.train < 5).sum())},
        "no test clip": {"gloss": int((gloss_classes.test == 0).sum()), "(gloss, variation)": int((variation_classes.test == 0).sum())},
    }
)

150 of 2000 glosses have more than one variation:
variation_id     1    2  3
glosses       1850  146  4


,classes,median train,min train,train < 5,no test clip
gloss,2000,7.0,5,0,0
"(gloss, variation)",2154,6.0,3,145,0


### Is a variation just a website?

If a variation came from one source site, a model could learn the site rather than the sign. For the
glosses with several variations: how many variations come from a single source or signer, and how
well the variation is predicted by the source alone, against always guessing the gloss's most common
variation.

In [3]:
multi = instances[instances.gloss.isin(per_gloss[per_gloss > 1].index)]
spread = multi.groupby(["gloss", "variation_id"]).agg(sources=("source", "nunique"), signers=("signer_id", "nunique"))
print(f"{len(spread)} variations: {int((spread.sources == 1).sum())} from a single source, {int((spread.signers == 1).sum())} from a single signer; median {spread.signers.median():.0f} signers")

def share_correct(frame: pd.DataFrame, by: list[str]) -> float:
    """Share of instances whose variation equals the most common one within `by` groups."""
    majority = frame.groupby(by).variation_id.transform(lambda v: v.mode().iloc[0])
    return float((frame.variation_id == majority).mean())

pd.Series(
    {
        "most common variation of the gloss": share_correct(multi, ["gloss"]),
        "most common variation of the gloss and source": share_correct(multi, ["gloss", "source"]),
    },
    name="instances matched",
).round(3)

304 variations: 0 from a single source, 0 from a single signer; median 6 signers


most common variation of the gloss               0.600
most common variation of the gloss and source    0.813
Name: instances matched, dtype: float64

## Merging glosses that are the same sign

Two `(gloss, variation)` classes are joined if they share at least `min_shared` clips. At one shared
clip, unrelated signs chain together, so the table also shows the largest component.

In [4]:
pair_counts = shared_clip_counts(instances)
print(f"{len(pair_counts)} class pairs share a clip; by number of clips shared:")
print(pd.Series(list(pair_counts.values())).value_counts().sort_index().head(7).rename("pairs").to_frame().T.to_string())

rows = []
for min_shared in (1, 2, 3):
    signs = merge_signs(instances, min_shared)
    sizes = signs.groupby(signs).ngroups
    nodes = instances.assign(sign=signs).groupby("sign").apply(lambda g: g[["gloss", "variation_id"]].drop_duplicates().shape[0], include_groups=False)
    rows.append({"min shared clips": min_shared, "signs": sizes, "classes merged away": len(variation_classes) - sizes, "largest merged sign (classes)": int(nodes.max())})
display(pd.DataFrame(rows).set_index("min shared clips"))
chain = merge_signs(instances, 1)
biggest = chain.value_counts().index[0]
sorted(set(instances[chain == biggest].gloss))

671 class pairs share a clip; by number of clips shared:
         1    2    3   4  5  6  7
pairs  333  191  102  34  9  1  1


,signs,classes merged away,largest merged sign (classes)
min shared clips,,,
1,1625,529,10
2,1863,291,6
3,2022,132,4


['hearing',
 'lecture',
 'presentation',
 'public',
 'say',
 'scold',
 'speak',
 'speech',
 'talk',
 'tell']

## What a cleaned dataset would leave

Drop copies (one instance per clip, kept in train if any copy is in train, else val, else test) and
merge classes that share at least 2 clips. Set sizes and class sizes before and after:

In [5]:
priority = {"train": 0, "val": 1, "test": 2}
cleaned = (
    instances.assign(sign=merge_signs(instances, 2), rank=instances.set.map(priority))
    .sort_values("rank")
    .drop_duplicates("md5")
)
sign_sizes = class_sizes(cleaned, ["sign"])
before = instances.set.value_counts()
after = cleaned.set.value_counts()
display(pd.DataFrame({"before": before, "after": after, "removed": before - after}).loc[["train", "val", "test"]])
pd.DataFrame(
    {
        "classes": [len(variation_classes), len(sign_sizes)],
        "train < 5": [int((variation_classes.train < 5).sum()), int((sign_sizes.train < 5).sum())],
        "train < 3": [int((variation_classes.train < 3).sum()), int((sign_sizes.train < 3).sum())],
        "no val clip": [int((variation_classes.val == 0).sum()), int((sign_sizes.val == 0).sum())],
        "no test clip": [int((variation_classes.test == 0).sum()), int((sign_sizes.test == 0).sum())],
    },
    index=["(gloss, variation), as is", "merged signs, deduplicated"],
)

,before,after,removed
set,,,
train,14296,13703,593
val,3920,3615,305
test,2879,2692,187


,classes,train < 5,train < 3,no val clip,no test clip
"(gloss, variation), as is",2154,145,0,0,0
"merged signs, deduplicated",1863,168,8,11,25


## Conclusions (as of 2026-10-15)

- **Variation classes are feasible from the existing data.** 150 of 2000 glosses have more than one
  variation (146 with 2, 4 with 3), giving 2154 classes instead of 2000. Every one has at least 3
  train and 1 test instance (median 6 train), but 145 have fewer than 5 train instances, against none
  for glosses.
- **The variation label is partly a website label.** No variation comes from a single source or
  signer (median 6 signers), but guessing a variation from the gloss and source matches 81% of
  instances against 60% from the gloss alone. A variation classifier could learn some of the site, so
  it needs a check (for instance, accuracy per source) before its variation results are believed.
- **Merging same-sign glosses needs a threshold and review.** Joining classes on a single shared clip
  chains unrelated signs (one merged sign spans `hearing`, `lecture`, `presentation`, `public`, `say`,
  `scold`, `speak`, `speech`, `talk`, `tell`). Requiring 2 shared clips merges 291 classes away (2154
  to 1863) with at most 6 classes in a sign; 3 clips merges 132 away. 338 class pairs share 2 or more
  clips, which is a list that can be reviewed by hand.
- **Deduplicating is mechanical but leaves some classes empty in a set.** Keeping one copy per clip
  (train first) removes 593 train, 305 val and 187 test instances, and no test clip is left in a
  leaked set. With the merge, 25 of 1863 classes have no test clip and 11 no val clip, and 168 have
  fewer than 5 train instances, so the splits would have to be remade rather than kept from WLASL.
- **Still not covered:** near-duplicates (re-encodes hash differently), so a clean test set is not
  proven by exact hashing alone, and a cleaned dataset is not comparable with published WLASL numbers.